# 🔬 멀티모달 Gemini 기반 증빙 위변조 탐지 실습
본 실습에서는 최신 구글 GenAI SDK와 Gemini 모델을 활용하여 영수증, 인보이스, 거래내역서 등 증빙 문서의 디지털 변조 흔적을 탐지하는 고정밀 포렌식 파이프라인을 구축합니다.

### 🎯 학습 목표
1. 최신 `google-genai` SDK 환경 구축 및 Vertex AI (IAM) / AI Studio 클라이언트 연동
2. 멀티모달 분석을 위한 대화형 이미지 업로드 환경 구축
3. 구조화된 출력(`Structured Outputs`) 제약 조건을 활용한 안정적인 포렌식 데이터 추출

In [ ]:
# 라이브러리 설치 (최신 공식 Google GenAI SDK 및 관련 도구)
!pip install -q google-genai pandas pillow matplotlib openpyxl
print("✅ 필수 라이브러리 설치가 완료되었습니다.")

## 🔑 STEP 2. 구글 제미나이(Gemini) 클라이언트 초기화
구글의 최신 GenAI SDK는 단일화된 인터페이스를 제공합니다. 
* **AI Studio 방식**: `API_KEY`에 키를 입력하면 작동합니다.
* **Vertex AI IAM 방식**: `PROJECT_ID`와 `LOCATION`을 입력하면 인프라 환경의 계정 권한(IAM)을 자동 인지하여 연동됩니다.

In [ ]:
import os
import getpass
from google import genai
from google.genai import types

# 1. 이전의 잘못된 객체 참조 제거 및 변수 초기화
if 'client' in globals():
    del client

# 2. 인증 설정 (Vertex AI IAM 방식 사용 시 API_KEY는 반드시 비워둡니다)
API_KEY = ""
PROJECT_ID = "claude-code-dev-492600"
LOCATION = "global"

def get_client():
    # API Key가 명시적으로 존재할 경우 AI Studio 모드
    if API_KEY.strip():
        print("🔑 Initializing AI Studio Client...")
        return genai.Client(api_key=API_KEY)

    # Project ID가 존재할 경우 Vertex AI IAM 모드 (API_KEY 생략)
    if PROJECT_ID.strip():
        print(f"🌐 Initializing Vertex AI Client (IAM) for project '{PROJECT_ID}'...")
        return genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

    # 둘 다 없으면 수동 입력
    key = getpass.getpass("GEMINI_API_KEY를 입력하세요: ")
    return genai.Client(api_key=key)

try:
    client = get_client()
    print("✅ Client initialized successfully.")
except Exception as e:
    print(f"❌ Initialization Error: {e}")

## 📤 STEP 3. 분석 대상 이미지 로드 및 전처리 시각화
코랩 환경에서는 브라우저 팝업을 통해 로컬 파일을 업로드할 수 있으며, 로컬 주피터 환경에서는 작업 디렉토리 또는 `허위서류 공유` 폴더 내부의 이미지를 자동으로 탐색합니다.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

image_paths = []

# 1. Google Colab / Colab Enterprise 환경에서의 대화형 파일 업로드 지원
try:
    from google.colab import files
    print("📤 실습할 위조 서류 이미지 파일들을 업로드해 주세요 (여러 개 동시 선택 가능)...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        image_paths.append(filename)
    print(f"✅ 업로드 완료된 파일 목록: {image_paths}")
except ImportError:
    # 2. 로컬 개발 환경 폴백: '허위서류 공유' 폴더 또는 현재 디렉토리 자동 탐색
    print("💻 로컬 실행 환경 감지. '허위서류 공유' 폴더 또는 현재 작업 디렉토리 내 이미지를 탐색합니다...")
    image_folder = "허위서류 공유"
    if os.path.exists(image_folder):
        found_files = glob.glob(f"{image_folder}/*.*")
        image_paths = sorted([p for p in found_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))])
    else:
        found_files = glob.glob("*.*")
        image_paths = sorted([p for p in found_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))])

if not image_paths:
    print("⚠️ 탐색되거나 업로드된 이미지 파일이 없습니다. 실습을 진행하려면 이미지 파일을 업로드하거나 '허위서류 공유' 폴더에 넣어주세요.")
else:
    print(f"📂 총 {len(image_paths)}개의 이미지 파일이 로드되었습니다:")
    for path in image_paths:
        print(f"  - {os.path.basename(path)}")

# 이미지 그리드 시각화 헬퍼 함수
def show_images_grid(paths):
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(20, 10))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        try:
            img = Image.open(p)
            ax.imshow(img)
            ax.set_title(os.path.basename(p), fontsize=10)
            ax.axis("off")
        except Exception as e:
            print(f"❌ {p} 파일 열기 실패: {e}")
    plt.tight_layout()
    plt.show()

if image_paths:
    show_images_grid(image_paths)

## 🧠 STEP 4. 포렌식 전문 시스템 프롬프트 및 분석 파이프라인 정의
글로벌 수준의 위조 분석 기준을 담은 **영어 시스템 프롬프트**를 정의합니다. LLM의 추론 및 논리 전개는 영어로 정밀하게 수행되지만, `response_schema` 규칙을 활용하여 최종 결과의 '이유(`reason`)'만 **한국어**로 출력하도록 제어합니다.

### 🔍 주요 포렌식 체크포인트
1. **논리적 모순**: 결제일, 승인번호, 바코드 간의 시간적/형식적 인과관계 분석
2. **시각적 변조**: 특정 글자 주변의 미세한 픽셀 깨짐(압축 노이즈), 폰트 자간/두께 불일치
3. **AI 생성**: 요소 검사(Inspect Element) 변조 후 캡처 흔적, 흐릿하거나 깨진 가짜 로고

In [ ]:
import json

# =====================================================================
# 1. High-Detail Forensic Document Analysis System Prompt (English)
# =====================================================================
FORENSIC_SYSTEM_PROMPT = """
You are an expert Forensic Document Examiner and Fraud Prevention Analyst specializing in documents such as receipts, invoices, tax statements, and official transaction screenshots.
Your sole mission is to analyze the provided document image to identify if it is faked, forged, altered, or digitally manipulated, and then output your final expert judgment into three specific fields.

### DETAILED INSPECTION PROTOCOLS
You must meticulously scan the document image for the following indicators of fraud:

1. Textual, Grammatical, and Logical Inconsistencies:
   - Date & Time Logic: Verify if the payment dates, approval dates, or order dates logically align with surrounding text, order numbers, approval codes, or any visible URL timestamps.
   - Typographical & Brand Frauds: Look for spelling mistakes, unnatural phrasing, or subtle typosquatting in famous store/brand names (especially checking for character level alterations in non-English texts like Korean).
   - Address Cross-Referencing: Check if the branch office name matches the format of the provided road-name or lot-number address.
   - URL/Domain Anomaly: Detect suspicious domains, phishing paths, or altered web address text in screenshots.

2. Visual and Digital Tampering Indicators:
   - Font & Border Discrepancies: Scan for uneven text alignments, inconsistent font weights, mixed typefaces within the same line, or unnatural pixel artifacts/compression noise around text borders.
   - Digital Composition & Cut-and-Paste: Look for differences in background noise, texture mismatches, or straight-line clipping artifacts around critical numbers (amounts, dates, approval numbers).
   - Inspect Element Modification: Check for unnaturally perfect system fonts layered over compressed image backgrounds, which indicates browser text alteration before capturing.

3. AI Layout and Synthetic Template Generation:
   - Synthetic Structural Templates: Identify unnatural spacing, missing dividing lines, or geometrically distorted grids.
   - Hallucinated Graphics: Spot distorted corporate logos, deformed barcodes, corrupted QR codes, or unreadable miniature text elements that are typical in AI-generated images.

### OUTPUT JSON REQUIREMENT
You must output a single, valid JSON object matching the requested schema exactly. 
All string explanations in the 'reason' field MUST be written in Korean, provided with specific forensic context.
"""

# =====================================================================
# 2. Execution & API Call Function (Structured Outputs)
# =====================================================================
def analyze_document_forgery(image_path: str, model_name: str = "gemini-3.5-flash") -> dict:
    if not os.path.exists(image_path):
        return {"error": f"파일을 찾을 수 없습니다: {image_path}"}
        
    try:
        img = Image.open(image_path)
        
        response = client.models.generate_content(
            model=model_name,
            contents=[img],
            config=types.GenerateContentConfig(
                system_instruction=FORENSIC_SYSTEM_PROMPT,
                response_mime_type="application/json",
                # Pydantic 선언 없이 config 내에서 강하게 JSON 구조체 제약 조건 바인딩
                response_schema={
                    "type": "OBJECT",
                    "properties": {
                        "is_forged": {
                            "type": "BOOLEAN", 
                            "description": "Final verdict. True if the document has any signs of forgery, alteration, or digital tampering. Otherwise, False."
                        },
                        "confidence_score": {
                            "type": "NUMBER", 
                            "description": "Confidence level of the verdict, ranging from 0.0 (no confidence) to 1.0 (absolute certainty)."
                        },
                        "reason": {
                            "type": "STRING", 
                            "description": "Detailed forensic reasoning and listed anomalies that led to this verdict. MUST BE WRITTEN IN KOREAN."
                        }
                    },
                    "required": ["is_forged", "confidence_score", "reason"]
                },
                temperature=0.1  # 포렌식 분석의 정밀도 및 결정론적 결과를 위해 최저 온도 설정
            )
        )
        # 반환된 순수 JSON 문자열을 즉시 파이썬 딕셔너리로 변환
        return json.loads(response.text)
        
    except Exception as e:
        return {"error": f"모델 실행 중 예외 발생: {str(e)}"}

## 🔬 STEP 5. 파이프라인 구동 및 위변조 벤치마크 결과 확인
앞서 업로드한 문서들을 순회하며 위변조 탐지 분석을 자동으로 일괄 수행합니다. 구조화된 출력 방식 덕분에 JSON 포맷이 깨지지 않고 완벽하게 출력됩니다.

In [ ]:
# =====================================================================
# 3. Multi-Image Evaluation Loop & Console Output (Korean Presentation)
# =====================================================================
if image_paths:
    print("🔬 고정밀 문서 위변조 탐지 파이프라인 가동...")
    
    for path in image_paths:
        file_name = os.path.basename(path)
        print(f"\n--------------------------------------------------\n📄 분석 파일: {file_name}")
        
        # 분석 알고리즘 실행 (기본 모델: gemini-3.5-flash)
        result = analyze_document_forgery(path, "gemini-3.5-flash")
        
        # 예외 및 에러 대응
        if "error" in result:
            print(f"  ❌ {result['error']}")
            continue
            
        # 직관적인 출력을 위한 데이터 리맵핑
        verdict_str = "⚠️ 위조 및 조작 의심 (FORGED)" if result["is_forged"] else "✅ 정상 증빙 문서 (GENUINE)"
        confidence_pct = result["confidence_score"] * 100
        
        print(f"  [최종 판정] {verdict_str}")
        print(f"  [확 신 도 ] {confidence_pct:.1f}%")
        print(f"  [상세 분석] {result['reason']}")
else:
    print("❌ 분석할 이미지가 없습니다. STEP 3을 확인해 주세요.")